<a href="https://colab.research.google.com/github/shay2222/ml-systems-lab/blob/main/03-triton-kernel/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Sun Oct 18 01:02:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q triton

Small GPU kernel in Triton and check whether it beats PyTorch on one operation. This is the experiment that shows low-level GPU work.

The operation is RMSNorm, a normalization step used in Qwen and many other LLMs. It’s short enough to read, and it’s a good target because PyTorch runs it as several separate GPU operations, so a fused kernel can do better.

In [6]:
import time

In [3]:
import torch
import triton
import triton.language as tl

@triton.jit
def rmsnorm_kernel(x_ptr, w_ptr, out_ptr, n_cols, eps, BLOCK: tl.constexpr):
    row = tl.program_id(0)                      # one GPU program per row
    offs = tl.arange(0, BLOCK)
    mask = offs < n_cols
    x = tl.load(x_ptr + row * n_cols + offs, mask=mask, other=0.0).to(tl.float32)
    var = tl.sum(x * x, axis=0) / n_cols        # mean of squares
    rstd = 1.0 / tl.sqrt(var + eps)
    w = tl.load(w_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    y = x * rstd * w
    tl.store(out_ptr + row * n_cols + offs, y.to(out_ptr.dtype.element_ty), mask=mask)

def rmsnorm_triton(x, w, eps=1e-6):
    n_rows, n_cols = x.shape
    out = torch.empty_like(x)
    BLOCK = triton.next_power_of_2(n_cols)
    rmsnorm_kernel[(n_rows,)](x, w, out, n_cols, eps, BLOCK=BLOCK)
    return out

def rmsnorm_torch(x, w, eps=1e-6):
    var = x.float().pow(2).mean(-1, keepdim=True)
    return (x.float() * torch.rsqrt(var + eps) * w.float()).to(x.dtype)

In [4]:
torch.manual_seed(0)
x = torch.randn(4096, 896, device="cuda", dtype=torch.float16)
w = torch.randn(896, device="cuda", dtype=torch.float16)

ref = rmsnorm_torch(x, w)
got = rmsnorm_triton(x, w)

max_diff = (ref.float() - got.float()).abs().max().item()
print(f"Max difference: {max_diff:.6f}")
print("PASS" if torch.allclose(ref.float(), got.float(), atol=1e-2, rtol=1e-2) else "FAIL")

Max difference: 0.003906
PASS


In [7]:
def bench(fn, x, w, iters=100):
    for _ in range(10):                          # warm-up
        fn(x, w)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(iters):
        fn(x, w)
    torch.cuda.synchronize()
    return (time.time() - start) / iters * 1000  # milliseconds per call

print("Rows x Cols | PyTorch (ms) | Triton (ms) | Speedup")
for rows, cols in [(1024, 896), (4096, 896), (16384, 896), (4096, 4096)]:
    x = torch.randn(rows, cols, device="cuda", dtype=torch.float16)
    w = torch.randn(cols, device="cuda", dtype=torch.float16)
    t_torch = bench(rmsnorm_torch, x, w)
    t_triton = bench(rmsnorm_triton, x, w)
    print(f"{rows}x{cols} | {t_torch:12.3f} | {t_triton:11.3f} | {t_torch / t_triton:.2f}x")

Rows x Cols | PyTorch (ms) | Triton (ms) | Speedup
1024x896 |        0.233 |       0.025 | 9.23x
4096x896 |        0.848 |       0.064 | 13.34x
16384x896 |        2.962 |       0.246 | 12.03x
4096x4096 |        3.304 |       0.295 | 11.19x
